# AGILAB data and pipeline-graph launch in Colab (PyPI)

This notebook shows the published-package AGILAB path for two built-in apps from Google Colab:

- `execution_pandas_project` for a data-worker-style workload.
- `uav_relay_queue_project` for a DAG-shaped pipeline with topology and queue artifacts.

It installs AGILAB from PyPI, installs the worker the first time, and runs both
paths locally through `AgiEnv` and `AGI.run(...)`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_data_dag_pypi.ipynb)


In [ ]:
# Published-package Colab path: install AGILAB from PyPI.
!python -m pip uninstall -y agilab agi-core agi-cluster agi-node agi-env >/dev/null 2>&1 || true
!python -m pip install -q uv agilab


In [ ]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

for bad_prefix in ("/content/agilab/src", "/content/agilab"):
    sys.path = [p for p in sys.path if p != bad_prefix and not p.startswith(bad_prefix + "/")]
for name in list(sys.modules):
    if name == "agilab" or name.startswith(("agilab.", "agi_env", "agi_env.", "agi_node", "agi_node.", "agi_cluster", "agi_cluster.")):
        del sys.modules[name]
importlib.invalidate_caches()

import agilab
from agi_cluster.agi_distributor import AGI
from agi_env import AgiEnv

APPS_PATH = Path(agilab.__file__).resolve().parent / "apps"
BUILTIN_ROOT = APPS_PATH / "builtin"


def worker_env_ready(app_env):
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if not worker_venv.exists():
        return False
    cmd = ["uv", "--quiet", "run", "--no-sync", "--project", str(worker_venv.parent)]
    pyvers_worker = getattr(app_env, "pyvers_worker", None)
    if pyvers_worker:
        cmd.extend(["--python", str(pyvers_worker)])
    cmd.extend(["python", "-c", "import agi_env, agi_node"])
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    return result.returncode == 0

async def install_if_needed(app_env, *, scheduler="127.0.0.1", workers=None, modes_enabled=0):
    if workers is None:
        workers = {"127.0.0.1": 1}
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if worker_env_ready(app_env):
        return False
    action = "Installing"
    if worker_venv.parent.exists():
        shutil.rmtree(worker_venv.parent, ignore_errors=True)
        action = "Reinstalling"
    print(f"{action} worker for {app_env.app}...")
    await AGI.install(app_env, scheduler=scheduler, workers=workers, modes_enabled=modes_enabled)
    return True

async def run_app(app_name: str):
    app_env = AgiEnv(apps_path=APPS_PATH, app=app_name, verbose=0)
    await install_if_needed(app_env, scheduler="127.0.0.1", workers={"127.0.0.1": 1}, modes_enabled=0)
    result = await AGI.run(
        app_env,
        scheduler="127.0.0.1",
        workers={"127.0.0.1": 1},
        mode=0,
    )
    log_root = Path.home() / "log" / "execute" / app_env.target
    print(f"\n=== {app_name} ===")
    print("Target:", app_env.target)
    print("Log root:", log_root)
    if log_root.exists():
        print("Files:", sorted(path.name for path in log_root.iterdir())[:20])
    return result

print("Installed apps path:", APPS_PATH)
print("Builtin root:", BUILTIN_ROOT)


In [ ]:
data_result = await run_app("execution_pandas_project")
data_result


In [ ]:
dag_result = await run_app("uav_relay_queue_project")
dag_result
